# Etiquetado automático mediante clustering

Notebook autónomo para Google Colab. Ejecuta todo el experimento: carga del dataset, preprocesamiento, clustering, asignación de etiquetas mediante apoyo experto, clasificación supervisada, métricas, visualizaciones y conclusión.

Las etiquetas reales se ocultan durante el clustering. Solo se emplean para simular el apoyo experto del 5 % o 10 % y para evaluar los resultados.

In [ ]:
# Colab ya incluye la mayoría de estas bibliotecas.
# Descomenta la línea siguiente si tu entorno no las tiene instaladas.
# !pip install -q numpy pandas scipy scikit-learn matplotlib

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, adjusted_rand_score, confusion_matrix,
                             davies_bouldin_score, f1_score, normalized_mutual_info_score,
                             precision_score, recall_score, roc_auc_score, silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings('ignore', category=ConvergenceWarning)
pd.set_option('display.float_format', lambda value: f'{value:.3f}')

## 1. Configuración experimental

Se repite el experimento con 10 semillas, una división estratificada 80/20 y dos cantidades de apoyo experto. La clase `0` del dataset corresponde a tumor maligno y es la clase positiva.

In [ ]:
TEST_SIZE = 0.20
SEEDS = tuple(range(101, 111))
SUPPORT_FRACTIONS = (0.05, 0.10)
POSITIVE_LABEL = 0  # 0 = malignant, 1 = benign
N_CLUSTERS = 2

ALGORITHMS = ('kmeans', 'hierarchical', 'gmm')
CLASSIFIERS = ('logistic_regression', 'svm_rbf')

ALGORITHM_LABELS = {
    'kmeans': 'K-Means',
    'hierarchical': 'Jerárquico (Ward)',
    'gmm': 'GMM',
}
CLASSIFIER_LABELS = {
    'logistic_regression': 'Regresión logística',
    'svm_rbf': 'SVM RBF',
}

## 2. Importación y exploración del dataset

`load_breast_cancer` contiene 569 observaciones, 30 variables morfológicas y el diagnóstico binario de referencia.

In [ ]:
dataset = load_breast_cancer()
X = dataset.data.astype(float)
y = dataset.target.astype(int)
feature_names = list(dataset.feature_names)

data_summary = pd.DataFrame({
    'indicador': ['Observaciones', 'Variables', 'Malignos (clase 0)', 'Benignos (clase 1)'],
    'valor': [X.shape[0], X.shape[1], int((y == 0).sum()), int((y == 1).sum())],
})
display(data_summary)
display(pd.DataFrame(X, columns=feature_names).head())

## 3. Funciones del experimento

La correspondencia entre clúster y clase se aprende exclusivamente con el subconjunto de apoyo experto. El algoritmo Húngaro encuentra la asignación uno-a-uno que maximiza la concordancia observada.

In [ ]:
def cluster_to_class_mapping(support_clusters, support_labels):
    """Crea el mapeo clúster-clase usando únicamente el apoyo experto."""
    counts = np.zeros((N_CLUSTERS, N_CLUSTERS), dtype=int)
    for cluster, label in zip(support_clusters, support_labels):
        counts[int(cluster), int(label)] += 1
    rows, columns = linear_sum_assignment(-counts)
    return {int(cluster): int(label) for cluster, label in zip(rows, columns)}


def apply_mapping(cluster_labels, mapping):
    return np.asarray([mapping[int(cluster)] for cluster in cluster_labels], dtype=int)


def fit_clusterer(name, X_train, seed):
    """Ajusta un clúster sin recibir etiquetas diagnósticas."""
    if name == 'kmeans':
        return KMeans(n_clusters=2, n_init=20, random_state=seed).fit_predict(X_train)
    if name == 'hierarchical':
        return AgglomerativeClustering(n_clusters=2, linkage='ward').fit_predict(X_train)
    if name == 'gmm':
        return GaussianMixture(n_components=2, covariance_type='full', random_state=seed).fit_predict(X_train)
    raise ValueError(f'Algoritmo no soportado: {name}')


def build_classifier(name, seed):
    if name == 'logistic_regression':
        return LogisticRegression(C=1.0, max_iter=5000, random_state=seed)
    if name == 'svm_rbf':
        return SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=seed)
    raise ValueError(f'Clasificador no soportado: {name}')


def score_classifier(name, X_train, y_train, X_test, y_test, seed):
    model = build_classifier(name, seed)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    positive_index = int(np.where(model.classes_ == POSITIVE_LABEL)[0][0])
    positive_scores = model.predict_proba(X_test)[:, positive_index]
    matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
    specificity = matrix[1, 1] / (matrix[1, 1] + matrix[1, 0])
    return {
        'accuracy': accuracy_score(y_test, predictions),
        'malignant_sensitivity': recall_score(y_test, predictions, pos_label=POSITIVE_LABEL, zero_division=0),
        'specificity': specificity,
        'precision': precision_score(y_test, predictions, pos_label=POSITIVE_LABEL, zero_division=0),
        'f1': f1_score(y_test, predictions, pos_label=POSITIVE_LABEL, zero_division=0),
        'roc_auc': roc_auc_score((y_test == POSITIVE_LABEL).astype(int), positive_scores),
    }


def aggregate(frame, group_columns, metric_columns):
    grouped = frame.groupby(group_columns)[metric_columns]
    summary = pd.concat([
        grouped.mean().add_suffix('_mean'),
        grouped.std(ddof=1).fillna(0).add_suffix('_std'),
        grouped.size().rename('n'),
    ], axis=1).reset_index()
    for metric in metric_columns:
        summary[f'{metric}_ci95'] = 1.96 * summary[f'{metric}_std'] / np.sqrt(summary['n'])
    return summary

## 4. Ejecución del experimento

Para cada semilla se separan entrenamiento y prueba, se escala solo el entrenamiento y se ajustan los tres métodos de clustering. Se comparan pseudoetiquetas, etiquetas reales completas y etiquetas reales del subconjunto de apoyo.

In [ ]:
cluster_rows = []
classifier_rows = []
visual_data = None

for seed in SEEDS:
    train_indices, test_indices = train_test_split(
        np.arange(len(y)), test_size=TEST_SIZE, stratify=y, random_state=seed
    )
    X_train_raw, X_test_raw = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    scaler = StandardScaler().fit(X_train_raw)
    X_train = scaler.transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    for classifier_name in CLASSIFIERS:
        classifier_rows.append({
            'seed': seed, 'algorithm': 'full_real', 'support_fraction': 1.0,
            'classifier': classifier_name, 'scheme': 'full_real',
            **score_classifier(classifier_name, X_train, y_train, X_test, y_test, seed),
        })

    support_indices_by_fraction = {}
    for fraction in SUPPORT_FRACTIONS:
        support_indices, _ = train_test_split(
            np.arange(len(y_train)), train_size=fraction, stratify=y_train, random_state=seed
        )
        support_indices = np.sort(support_indices)
        support_indices_by_fraction[fraction] = support_indices

        for classifier_name in CLASSIFIERS:
            classifier_rows.append({
                'seed': seed, 'algorithm': 'support_real', 'support_fraction': fraction,
                'classifier': classifier_name, 'scheme': 'support_real',
                **score_classifier(
                    classifier_name, X_train[support_indices], y_train[support_indices],
                    X_test, y_test, seed
                ),
            })

    clusters_by_algorithm = {}
    for algorithm in ALGORITHMS:
        cluster_labels = fit_clusterer(algorithm, X_train, seed)
        clusters_by_algorithm[algorithm] = cluster_labels
        optimal_mapping = cluster_to_class_mapping(cluster_labels, y_train)
        optimal_accuracy = np.mean(apply_mapping(cluster_labels, optimal_mapping) == y_train)

        for fraction in SUPPORT_FRACTIONS:
            support_indices = support_indices_by_fraction[fraction]
            support_mapping = cluster_to_class_mapping(
                cluster_labels[support_indices], y_train[support_indices]
            )
            pseudo_labels = apply_mapping(cluster_labels, support_mapping)

            cluster_rows.append({
                'seed': seed, 'algorithm': algorithm, 'support_fraction': fraction,
                'silhouette': silhouette_score(X_train, cluster_labels),
                'davies_bouldin': davies_bouldin_score(X_train, cluster_labels),
                'cluster_accuracy_optimal': optimal_accuracy,
                'ari': adjusted_rand_score(y_train, cluster_labels),
                'nmi': normalized_mutual_info_score(y_train, cluster_labels),
                'labeling_accuracy_support_mapping': np.mean(pseudo_labels == y_train),
            })

            for classifier_name in CLASSIFIERS:
                classifier_rows.append({
                    'seed': seed, 'algorithm': algorithm, 'support_fraction': fraction,
                    'classifier': classifier_name, 'scheme': 'generated',
                    **score_classifier(classifier_name, X_train, pseudo_labels, X_test, y_test, seed),
                })

    if visual_data is None:
        visual_data = {
            'seed': seed, 'X_train': X_train, 'y_train': y_train,
            'clusters': clusters_by_algorithm,
        }

cluster_results = pd.DataFrame(cluster_rows)
classifier_results = pd.DataFrame(classifier_rows)
print(f'Filas de clustering: {len(cluster_results)}')
print(f'Filas de clasificación: {len(classifier_results)}')

## 5. Calidad de los agrupamientos

Silhouette y Davies-Bouldin miden estructura interna; exactitud óptima, ARI y NMI comparan los clústeres con el diagnóstico oculto solo después de ajustarlos.

In [ ]:
cluster_metrics = [
    'silhouette', 'davies_bouldin', 'cluster_accuracy_optimal', 'ari', 'nmi',
    'labeling_accuracy_support_mapping',
]
cluster_summary = aggregate(cluster_results, ['algorithm', 'support_fraction'], cluster_metrics)
cluster_summary['algoritmo'] = cluster_summary['algorithm'].map(ALGORITHM_LABELS)
display(cluster_summary[[
    'algoritmo', 'support_fraction', 'silhouette_mean', 'davies_bouldin_mean',
    'cluster_accuracy_optimal_mean', 'ari_mean', 'nmi_mean',
    'labeling_accuracy_support_mapping_mean',
]].sort_values(['cluster_accuracy_optimal_mean', 'support_fraction'], ascending=[False, True]))

In [ ]:
X_2d = PCA(n_components=2, random_state=visual_data['seed']).fit_transform(visual_data['X_train'])
figure, axes = plt.subplots(1, 4, figsize=(20, 4), constrained_layout=True)
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=visual_data['y_train'], cmap='coolwarm', s=16)
axes[0].set_title('Diagnóstico real\n(solo evaluación)')

for axis, algorithm in zip(axes[1:], ALGORITHMS):
    axis.scatter(X_2d[:, 0], X_2d[:, 1], c=visual_data['clusters'][algorithm], cmap='viridis', s=16)
    axis.set_title(ALGORITHM_LABELS[algorithm])

for axis in axes:
    axis.set_xlabel('PCA 1')
    axis.set_ylabel('PCA 2')

figure.suptitle(f'Visualización PCA del entrenamiento - semilla {visual_data["seed"]}')
plt.show()

## 6. Desempeño de clasificación y regla de viabilidad

Una combinación es viable cuando la exactitud de etiquetado es al menos 90% y las brechas absolutas de exactitud y sensibilidad maligna, frente al modelo con todas las etiquetas reales, no superan 5 puntos porcentuales.

In [ ]:
classifier_metrics = ['accuracy', 'malignant_sensitivity', 'specificity', 'precision', 'f1', 'roc_auc']
classifier_summary = aggregate(
    classifier_results,
    ['algorithm', 'support_fraction', 'classifier', 'scheme'],
    classifier_metrics,
)

generated = classifier_summary[classifier_summary['scheme'] == 'generated'].copy()
full_real = classifier_summary[classifier_summary['scheme'] == 'full_real'][
    ['classifier', 'accuracy_mean', 'malignant_sensitivity_mean']
].rename(columns={
    'accuracy_mean': 'full_accuracy_mean',
    'malignant_sensitivity_mean': 'full_malignant_sensitivity_mean',
})

viability = cluster_summary.merge(
    generated[['algorithm', 'support_fraction', 'classifier', 'accuracy_mean', 'malignant_sensitivity_mean']],
    on=['algorithm', 'support_fraction'],
).merge(full_real, on='classifier')

viability['accuracy_gap'] = viability['full_accuracy_mean'] - viability['accuracy_mean']
viability['sensitivity_gap'] = (
    viability['full_malignant_sensitivity_mean'] - viability['malignant_sensitivity_mean']
)
viability['viable'] = (
    (viability['labeling_accuracy_support_mapping_mean'] >= 0.90)
    & (viability['accuracy_gap'].abs() <= 0.05)
    & (viability['sensitivity_gap'].abs() <= 0.05)
)

viability['algoritmo'] = viability['algorithm'].map(ALGORITHM_LABELS)
viability['clasificador'] = viability['classifier'].map(CLASSIFIER_LABELS)
display(viability[[
    'algoritmo', 'support_fraction', 'clasificador',
    'labeling_accuracy_support_mapping_mean', 'accuracy_mean',
    'malignant_sensitivity_mean', 'accuracy_gap', 'sensitivity_gap', 'viable',
]].sort_values(['viable', 'accuracy_mean'], ascending=[False, False]))

In [ ]:
ranking = generated.sort_values(['accuracy_mean', 'malignant_sensitivity_mean'], ascending=False).copy()
ranking['label'] = ranking.apply(
    lambda row: f"{ALGORITHM_LABELS[row['algorithm']]}\n{int(row['support_fraction'] * 100)}% - {CLASSIFIER_LABELS[row['classifier']]}",
    axis=1,
)

plt.figure(figsize=(12, 5))
plt.bar(ranking['label'], ranking['accuracy_mean'], yerr=ranking['accuracy_ci95'], capsize=4, color='#386cb0')
plt.ylim(0, 1.05)
plt.ylabel('Exactitud media en prueba')
plt.title('Clasificadores entrenados con pseudoetiquetas')
plt.xticks(rotation=40, ha='right')
plt.show()

display(ranking[[
    'algorithm', 'support_fraction', 'classifier', 'accuracy_mean', 'accuracy_ci95',
    'malignant_sensitivity_mean', 'malignant_sensitivity_ci95',
]])

## 7. Conclusión automática

La conclusión se calcula a partir de la regla de viabilidad y no está escrita manualmente.

In [ ]:
viable_results = viability[viability['viable']].sort_values('accuracy_mean', ascending=False)
best = ranking.iloc[0]

print(f'Combinaciones viables: {len(viable_results)} de {len(viability)}')
print(
    f"Mejor combinación: {ALGORITHM_LABELS[best['algorithm']]} + "
    f"{CLASSIFIER_LABELS[best['classifier']]} con {int(best['support_fraction'] * 100)}% de apoyo experto."
)
print(f"Exactitud media: {best['accuracy_mean']:.3f}; sensibilidad maligna: {best['malignant_sensitivity_mean']:.3f}.")

if len(viable_results):
    print('Conclusión: al menos un enfoque satisface los criterios definidos para este dataset.')
else:
    print('Conclusión: ningún enfoque satisface simultáneamente los criterios definidos.')

print('Limitación: los resultados no sustituyen un diagnóstico clínico ni se generalizan automáticamente a otros contextos.')

## 8. Guardar resultados opcionalmente

Esta celda guarda las tablas generadas. En Colab puedes comprimir la carpeta y descargarla.

In [ ]:
output_dir = Path('resultados_colab')
output_dir.mkdir(exist_ok=True)
cluster_results.to_csv(output_dir / 'cluster_results.csv', index=False)
classifier_results.to_csv(output_dir / 'classifier_results.csv', index=False)
cluster_summary.to_csv(output_dir / 'cluster_summary.csv', index=False)
classifier_summary.to_csv(output_dir / 'classifier_summary.csv', index=False)
viability.to_csv(output_dir / 'viability.csv', index=False)
ranking.to_csv(output_dir / 'ranking.csv', index=False)
print(f'Resultados guardados en: {output_dir.resolve()}')

# En Google Colab, descomenta estas líneas para descargar un ZIP:
# !zip -r resultados_colab.zip resultados_colab
# from google.colab import files
# files.download('resultados_colab.zip')